<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula06a_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip titanic.zip

Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [2]:
import pandas as pd

df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [11]:
y = df['Survived']
X = df.drop('Survived', axis=1)

In [12]:
len(y)

891

In [13]:
for col in X.columns:
  print(f"{col:>12} {len(set(X[col])):4} {X[col].dtype}")

 PassengerId  891 int64
      Pclass    3 int64
        Name  891 object
         Sex    2 object
         Age  265 float64
       SibSp    7 int64
       Parch    7 int64
      Ticket  681 object
        Fare  248 float64
       Cabin  148 object
    Embarked    4 object


In [14]:
caracterriscas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
Xdrop = X.drop(caracterriscas_indesejadas, axis=1)
Xdrop.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [16]:
Xnum = Xdrop.select_dtypes(include='number')
Xnum.columns

Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [17]:
for col in Xnum.columns:
  print(f"{col:>12} {X[col].isnull().sum():4}")

      Pclass    0
         Age  177
       SibSp    0
       Parch    0
        Fare    0


In [19]:
Xnum[-5:]

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [20]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
XnumTratado = imputer.fit_transform(Xnum)
XnumTratado[-5:]

array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75]])

In [21]:
Xcat = Xdrop.select_dtypes(object)
Xcat.columns

Index(['Sex', 'Embarked'], dtype='object')

In [22]:
for col in Xcat.columns:
  print(f"{col:>12} {X[col].isnull().sum():4}")

         Sex    0
    Embarked    2


In [23]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
XcatTratado = imputer.fit_transform(Xcat)
XcatTratado[-5:]

array([['male', 'S'],
       ['female', 'S'],
       ['female', 'S'],
       ['male', 'C'],
       ['male', 'Q']], dtype=object)

In [24]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
XcatTratadoHot = encoder.fit_transform(XcatTratado)
XcatTratadoHot

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1782 stored elements and shape (891, 5)>

In [25]:
XcatTratadoHot.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [26]:
import numpy as np
Xtratado = np.c_[XnumTratado, XcatTratadoHot.toarray()]
print(Xtratado.shape)
Xtratado[-5:]

(891, 10)


array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ,  0.  ,  1.  ,  0.  ,  0.  ,
         1.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ,  1.  ,  0.  ,  0.  ,  0.  ,
         1.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45,  1.  ,  0.  ,  0.  ,  0.  ,
         1.  ],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ,  0.  ,  1.  ,  1.  ,  0.  ,
         0.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75,  0.  ,  1.  ,  0.  ,  1.  ,
         0.  ]])

In [27]:
from sklearn.base import BaseEstimator, TransformerMixin

class AtributosDesejados(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.colunas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.drop(self.colunas_indesejadas, axis=1)

atributos_desejados = AtributosDesejados()
Xdrop = atributos_desejados.fit_transform(X)
Xdrop.tail()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
886,2,male,27.0,0,0,13.00,S
887,1,female,19.0,0,0,30.00,S
888,3,female,NaN,1,2,23.45,S
889,1,male,26.0,0,0,30.00,C
890,3,male,32.0,0,0,7.75,Q


In [28]:
class AtributosNumericos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.select_dtypes(include='number')

atributos_numericos = AtributosNumericos()
Xnum = atributos_numericos.fit_transform(Xdrop)
Xnum.tail()

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipenum = Pipeline([
    ('atributos_numericos', AtributosNumericos()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
XnumTratado = pipenum.fit_transform(Xdrop)
XnumTratado[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783]])

In [32]:
class AtributosCategoricos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.select_dtypes(object)

atributos_categoricos = AtributosCategoricos()
Xcat = atributos_categoricos.fit_transform(Xdrop)
Xcat.tail()

,Sex,Embarked
886,male,S
887,female,S
888,female,S
889,male,C
890,male,Q


In [33]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pipecat = Pipeline([
    ('atributos_categoricos', AtributosCategoricos()),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])
XcatTratado = pipecat.fit_transform(Xdrop)
XcatTratado.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [35]:
from sklearn.pipeline import FeatureUnion

unecaracteristicas = FeatureUnion([
    ('pipenum', pipenum),
    ('pipecat', pipecat)
])
Xtratado = unecaracteristicas.fit_transform(Xdrop)
Xtratado[-5:].toarray()[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104,
         0.        ,  1.        ,  1.        ,  0.        ,  0.        ],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ]])

In [36]:
class DenseTransformer(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None, **fit_params):
    return self
  def transform(self, X, y=None, **fit_params):
    return X.toarray()

preproc = Pipeline([
    ('atributos_desejados', AtributosDesejados()),
    ('unecaracteristicas', unecaracteristicas),
    ('densidade', DenseTransformer())
])
Xtratado = preproc.fit_transform(X)
Xtratado[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104,
         0.        ,  1.        ,  1.        ,  0.        ,  0.        ],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ]])

In [37]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([
    ('preproc', preproc),
    ('classificador', RandomForestClassifier())
])
clf.fit(X, y)

Pipeline(steps=[('preproc',
                 Pipeline(steps=[('atributos_desejados', AtributosDesejados()),
                                 ('unecaracteristicas',
                                  FeatureUnion(transformer_list=[('pipenum',
                                                                  Pipeline(steps=[('atributos_numericos',
                                                                                   AtributosNumericos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='median')),
                                                                                  ('scaler',
                                                                                   StandardScaler())])),
                                                                 ('pipecat',
                                                                  Pipeline(steps=[('atributos_categoricos',
                                                                                   AtributosCategoricos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('encoder',
                                                                                   OneHotEncoder())]))])),
                                 ('densidade', DenseTransformer())])),
                ('classificador', RandomForestClassifier())])

In [39]:
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

y_pred = clf.predict(X)
accuracy_score(y, y_pred)

0.9797979797979798

In [40]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X, y, cv=5)
print(scores.mean())
print(scores)

0.8058690603226413
[0.77653631 0.81460674 0.83707865 0.7752809  0.8258427 ]


In [41]:
Xtest = pd.read_csv('test.csv')
Xtest.shape

(418, 11)

In [42]:
y_pred = clf.predict(Xtest)
y_pred

array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [44]:
!head gender_submission.csv

PassengerId,Survived
892,0
893,1
894,0
895,0
896,1
897,0
898,1
899,0
900,1


In [45]:
submission = pd.read_csv('gender_submission.csv')
submission['Survived'] = y_pred
submission.to_csv('submission.csv', index=False)

In [46]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

estimators = [
    ('log_reg', LogisticRegression()),
    ('sgd', SGDClassifier()),
    ('knn', KNeighborsClassifier()),
    ('tree', DecisionTreeClassifier()),
    ('rf', RandomForestClassifier()),
    ('et', ExtraTreesClassifier()),
    ('nb', GaussianNB()),
    ('svm', SVC()),
    ('mlp', MLPClassifier()),
    ('lda', LinearDiscriminantAnalysis())
]


results = {
    'model': [],
    'accuracy_mean': [],
    'accuracy_std': []
}

for est in estimators:
  clf = Pipeline([
      ('preproc', preproc),
      ('classificador', est[1])
  ])
  scores = cross_val_score(clf, X, y, cv=5)
  results['model'].append(est[0])
  results['accuracy_mean'].append(scores.mean())
  results['accuracy_std'].append(scores.std())
  print(f"{est[0]:>5} {scores.mean():.2f} {scores.std():.2f}")


log_reg 0.79 0.02
  sgd 0.70 0.09
  knn 0.81 0.02
 tree 0.77 0.03
   rf 0.81 0.02
   et 0.79 0.03
   nb 0.79 0.01
  svm 0.83 0.02
  mlp 0.83 0.03
  lda 0.79 0.02


In [47]:
!pip install optuna -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.7 MB/s eta 0:00:00


In [49]:
from multiprocessing import Pipe
from sklearn.base import BaseEstimator, ClassifierMixin
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

class OptunaSVM(BaseEstimator, ClassifierMixin):
  def __init__(self, n_trials=20):
    self.n_trials = n_trials
  def fit(self, X, y=None):
    def objective(trial):
      C = trial.suggest_float('C', 1e-10, 1e10, log=True)
      gamma = trial.suggest_float('gamma', 1e-10, 1e10, log=True)
      clf = SVC(C=C, gamma=gamma)
      pipe = Pipeline([
          ('preproc', preproc),
          ('classificador', clf)
      ])
      scores = cross_val_score(pipe, X, y, cv=5)
      return scores.mean()
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=self.n_trials)
    self.model = Pipeline([
        ('preproc', preproc),
        ('classificador', SVC(**study.best_params))
    ])
    self.model.fit(X, y)
    return self
  def predict(self, X, y=None):
    return self.model.predict(X)
  def score(self, X, y):
    return self.model.score(X, y)

model = OptunaSVM(n_trials=20)
scores = cross_val_score(model, X, y, cv=5)
print(scores.mean())
print(scores)

[I 2026-09-28 23:37:30,396] A new study created in memory with name: no-name-d0a6febf-f179-46e6-bdfe-8d940b3cbcd5
[I 2026-09-28 23:41:05,980] Trial 0 finished with value: 0.7430808627991726 and parameters: {'C': 39349881.29703093, 'gamma': 0.2030894744093545}. Best is trial 0 with value: 0.7430808627991726.
[I 2026-09-28 23:41:09,706] Trial 1 finished with value: 0.8173643258150299 and parameters: {'C': 6839416217.748378, 'gamma': 5.332026525126929e-05}. Best is trial 1 with value: 0.8173643258150299.
[I 2026-09-28 23:41:10,020] Trial 2 finished with value: 0.6137496306510392 and parameters: {'C': 79864.33972411069, 'gamma': 1063951.2243997543}. Best is trial 1 with value: 0.8173643258150299.
[I 2026-09-28 23:41:10,357] Trial 3 finished with value: 0.6137496306510392 and parameters: {'C': 13983160.578798756, 'gamma': 673178.2168024845}. Best is trial 1 with value: 0.8173643258150299.
[I 2026-09-28 23:41:10,578] Trial 4 finished with value: 0.8005318625036935 and parameters: {'C': 205.0

0.8024543343167411
[0.81564246 0.81460674 0.78089888 0.76404494 0.83707865]


In [51]:
model = OptunaSVM(n_trials=30)
model.fit(X, y)
y_pred = model.predict(Xtest)
submission = pd.read_csv('gender_submission.csv')
submission['Survived'] = y_pred
submission.to_csv('submission_optuna.csv', index=False)

[I 2026-09-28 23:55:51,105] A new study created in memory with name: no-name-c572e23d-c3fb-4650-bf8d-1f61e230ace8
[I 2026-09-28 23:55:51,418] Trial 0 finished with value: 0.6374678300169482 and parameters: {'C': 173325.3948836307, 'gamma': 3730518.4114332926}. Best is trial 0 with value: 0.6374678300169482.
[I 2026-09-28 23:55:54,346] Trial 1 finished with value: 0.6161257924800703 and parameters: {'C': 437183721.2190712, 'gamma': 1873.6150146031034}. Best is trial 0 with value: 0.6374678300169482.
[I 2026-09-28 23:55:54,596] Trial 2 finished with value: 0.6385851484527023 and parameters: {'C': 5.895898047237088, 'gamma': 762548.5083071947}. Best is trial 2 with value: 0.6385851484527023.
[I 2026-09-29 00:09:03,501] Trial 3 finished with value: 0.6644466762915071 and parameters: {'C': 1358670923.3326426, 'gamma': 0.8167131659641242}. Best is trial 3 with value: 0.6644466762915071.
[I 2026-09-29 00:09:03,777] Trial 4 finished with value: 0.6374615529470844 and parameters: {'C': 30.19332

In [55]:
class OptunaMLP(BaseEstimator, ClassifierMixin):
  def __init__(self, n_trials=20):
    self.n_trials = n_trials
  def fit(self, X, y=None):
    def objective(trial):
      hidden_layer_size = trial.suggest_int('hidden_layer_size', 50, 500)
      max_iter = trial.suggest_int('max_iter', 100, 1000)
      clf = MLPClassifier(hidden_layer_sizes=(hidden_layer_size, ), max_iter=max_iter)
      pipe = Pipeline([
          ('preproc', preproc),
          ('classificador', clf)
      ])
      scores = cross_val_score(pipe, X, y, cv=5)
      return scores.mean()
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=self.n_trials)
    print(study.best_params)
    self.model = Pipeline([
        ('preproc', preproc),
        ('classificador', MLPClassifier(
            hidden_layer_sizes=(study.best_params["hidden_layer_size"], ),
            max_iter=study.best_params["max_iter"]))
    ])
    self.model.fit(X, y)
    return self
  def predict(self, X, y=None):
    return self.model.predict(X)
  def score(self, X, y):
    return self.model.score(X, y)

model = OptunaMLP(n_trials=10)
scores = cross_val_score(model, X, y, cv=5)
print(scores.mean())
print(scores)

[I 2026-09-29 00:38:00,565] A new study created in memory with name: no-name-87e7e6c2-013f-4693-9253-2f5ed5095d27
[I 2026-09-29 00:38:02,577] Trial 0 finished with value: 0.8244164286417807 and parameters: {'hidden_layer_size': 126, 'max_iter': 212}. Best is trial 0 with value: 0.8244164286417807.
[I 2026-09-29 00:38:11,275] Trial 1 finished with value: 0.8132177681473456 and parameters: {'hidden_layer_size': 288, 'max_iter': 467}. Best is trial 0 with value: 0.8244164286417807.
[I 2026-09-29 00:38:17,542] Trial 2 finished with value: 0.8075544174135724 and parameters: {'hidden_layer_size': 97, 'max_iter': 842}. Best is trial 0 with value: 0.8244164286417807.
[I 2026-09-29 00:38:19,715] Trial 3 finished with value: 0.8202009258347287 and parameters: {'hidden_layer_size': 109, 'max_iter': 304}. Best is trial 0 with value: 0.8244164286417807.
[I 2026-09-29 00:38:21,745] Trial 4 finished with value: 0.8216093765389539 and parameters: {'hidden_layer_size': 253, 'max_iter': 165}. Best is tr

{'hidden_layer_size': 66, 'max_iter': 508}


[I 2026-09-29 00:38:58,945] A new study created in memory with name: no-name-ae2ca018-c336-4a4f-8f7c-13c920a2a1d4
[I 2026-09-29 00:39:01,890] Trial 0 finished with value: 0.8176598049837487 and parameters: {'hidden_layer_size': 82, 'max_iter': 446}. Best is trial 0 with value: 0.8176598049837487.
[I 2026-09-29 00:39:10,826] Trial 1 finished with value: 0.8022259430710135 and parameters: {'hidden_layer_size': 201, 'max_iter': 838}. Best is trial 0 with value: 0.8176598049837487.
[I 2026-09-29 00:39:30,113] Trial 2 finished with value: 0.8050132965625924 and parameters: {'hidden_layer_size': 401, 'max_iter': 676}. Best is trial 0 with value: 0.8176598049837487.
[I 2026-09-29 00:39:50,190] Trial 3 finished with value: 0.800817492366788 and parameters: {'hidden_layer_size': 365, 'max_iter': 952}. Best is trial 0 with value: 0.8176598049837487.
[I 2026-09-29 00:39:56,005] Trial 4 finished with value: 0.819078104993598 and parameters: {'hidden_layer_size': 474, 'max_iter': 160}. Best is tria

{'hidden_layer_size': 474, 'max_iter': 160}


[I 2026-09-29 00:40:44,196] A new study created in memory with name: no-name-c682b872-a1f5-46f1-a3f3-54aaf86e919a
[I 2026-09-29 00:41:07,561] Trial 0 finished with value: 0.7854427262877968 and parameters: {'hidden_layer_size': 388, 'max_iter': 947}. Best is trial 0 with value: 0.7854427262877968.
[I 2026-09-29 00:41:14,795] Trial 1 finished with value: 0.8022653402935094 and parameters: {'hidden_layer_size': 182, 'max_iter': 738}. Best is trial 1 with value: 0.8022653402935094.
[I 2026-09-29 00:41:43,373] Trial 2 finished with value: 0.7854328769821727 and parameters: {'hidden_layer_size': 495, 'max_iter': 947}. Best is trial 1 with value: 0.8022653402935094.
[I 2026-09-29 00:42:02,236] Trial 3 finished with value: 0.7826356741849699 and parameters: {'hidden_layer_size': 386, 'max_iter': 794}. Best is trial 1 with value: 0.8022653402935094.
[I 2026-09-29 00:42:04,159] Trial 4 finished with value: 0.8149019994090416 and parameters: {'hidden_layer_size': 220, 'max_iter': 147}. Best is t

{'hidden_layer_size': 231, 'max_iter': 389}


[I 2026-09-29 00:42:36,568] A new study created in memory with name: no-name-61d655b5-e803-4830-9015-5c01265cadcc
[I 2026-09-29 00:42:46,027] Trial 0 finished with value: 0.8078892938047868 and parameters: {'hidden_layer_size': 215, 'max_iter': 826}. Best is trial 0 with value: 0.8078892938047868.
[I 2026-09-29 00:42:47,414] Trial 1 finished with value: 0.8401457697232345 and parameters: {'hidden_layer_size': 92, 'max_iter': 198}. Best is trial 1 with value: 0.8401457697232345.
[I 2026-09-29 00:43:06,515] Trial 2 finished with value: 0.8134935487048164 and parameters: {'hidden_layer_size': 464, 'max_iter': 643}. Best is trial 1 with value: 0.8401457697232345.
[I 2026-09-29 00:43:13,618] Trial 3 finished with value: 0.8135033980104402 and parameters: {'hidden_layer_size': 138, 'max_iter': 865}. Best is trial 1 with value: 0.8401457697232345.
[I 2026-09-29 00:43:16,734] Trial 4 finished with value: 0.8387373190190092 and parameters: {'hidden_layer_size': 236, 'max_iter': 236}. Best is tr

{'hidden_layer_size': 92, 'max_iter': 198}


[I 2026-09-29 00:44:07,010] A new study created in memory with name: no-name-8b9b0a96-5ead-4f82-a274-3bb45f663674
[I 2026-09-29 00:44:09,765] Trial 0 finished with value: 0.7952033881611346 and parameters: {'hidden_layer_size': 353, 'max_iter': 122}. Best is trial 0 with value: 0.7952033881611346.
[I 2026-09-29 00:44:12,549] Trial 1 finished with value: 0.7910174332709543 and parameters: {'hidden_layer_size': 188, 'max_iter': 266}. Best is trial 0 with value: 0.7952033881611346.
[I 2026-09-29 00:44:17,255] Trial 2 finished with value: 0.796621688170984 and parameters: {'hidden_layer_size': 182, 'max_iter': 396}. Best is trial 2 with value: 0.796621688170984.
[I 2026-09-29 00:44:19,435] Trial 3 finished with value: 0.7994090416625628 and parameters: {'hidden_layer_size': 224, 'max_iter': 189}. Best is trial 3 with value: 0.7994090416625628.
[I 2026-09-29 00:44:26,133] Trial 4 finished with value: 0.7882103811681276 and parameters: {'hidden_layer_size': 113, 'max_iter': 895}. Best is tri

{'hidden_layer_size': 328, 'max_iter': 143}
0.8227167158370472
[0.78212291 0.8258427  0.83146067 0.78089888 0.89325843]


In [56]:
model = OptunaMLP(n_trials=30)
model.fit(X, y)
y_pred = model.predict(Xtest)
submission = pd.read_csv('gender_submission.csv')
submission['Survived'] = y_pred
submission.to_csv('submission_optuna_mlp.csv', index=False)

[I 2026-09-29 00:45:24,945] A new study created in memory with name: no-name-b4810cbf-047e-4668-82fb-d9229faa0f01
[I 2026-09-29 00:45:29,739] Trial 0 finished with value: 0.8126043562864854 and parameters: {'hidden_layer_size': 172, 'max_iter': 340}. Best is trial 0 with value: 0.8126043562864854.
[I 2026-09-29 00:45:49,430] Trial 1 finished with value: 0.8092335697696316 and parameters: {'hidden_layer_size': 391, 'max_iter': 531}. Best is trial 0 with value: 0.8126043562864854.
[I 2026-09-29 00:46:10,933] Trial 2 finished with value: 0.8058627832527776 and parameters: {'hidden_layer_size': 383, 'max_iter': 899}. Best is trial 0 with value: 0.8126043562864854.
[I 2026-09-29 00:46:36,779] Trial 3 finished with value: 0.8025359362249702 and parameters: {'hidden_layer_size': 492, 'max_iter': 756}. Best is trial 0 with value: 0.8126043562864854.
[I 2026-09-29 00:46:43,933] Trial 4 finished with value: 0.8159625886636119 and parameters: {'hidden_layer_size': 480, 'max_iter': 178}. Best is t

{'hidden_layer_size': 290, 'max_iter': 108}
